# 05 — Összefoglaló dashboard: a számok egy helyen

**MIT:** Ez a fejezet a 01–04. fejezetek eredményeit foglalja egyetlen, interaktív felületre: a fő becsült hatások, a hatásmátrix, a kettős hatásgörbe és egy diagram- és térképgaléria.

**MIÉRT:** A korábbi fejezetek egymásra épülő láncot alkotnak (adat → térbeli mérés → árhatás → robusztusság). Ez az oldal a lánc VÉGEREDMÉNYÉT mutatja — minden szám ugyanabból a cache-elt, egyszer kiszámolt elemzési rétegből jön, ezért mindenhol azonosak.

**HOGYAN:** A notebook kizárólag az `analyses` modul cache-elt eredményeit olvassa (újraszámítás nélkül), és azokat rendezi KPI-kártyákba, táblákba és ábrákba. A galéria a statikus HTML-exportban is működő JS-választókkal mutatja az összes fő diagramtípust.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')
AREA_NEV = get_area_metadata(AREA)['name']
df = load_szamitott_master()
p = A.pontos_elado()

# ==== A cache-elt eredmények betöltése (nincs újraszámítás) ====
canon = A.canonical_hedonic()
grad = A.immission_gradient()
sar = A.sar_sem()
ml = A.ml_analysis()
rg = A.rent_gap_analysis()
mc = A.monte_carlo_risk()
lvc = A.lvc_analysis()
spat = A.spatial_metrics()

_zaj_tab = canon['zaj']
_teljes_tab = canon['teljes']
_meta = canon['meta']
_ref = str(_meta['ref_zona'].iloc[0])
_zone_cols = [c for c in _zaj_tab['valtozo'] if c.startswith(('<', '1', '3', '5')) and ' m' in c]
_zaj_col = '<150 m' if '<150 m' in _zaj_tab['valtozo'].values else _zone_cols[0]
_tod_cols = [c for c in _teljes_tab['valtozo'] if c.startswith('kp_')]
print(f"Cache-betöltés kész — {AREA_NEV}: N={int(_meta['n'].iloc[0])}, referencia-zóna: {_ref}")

In [ ]:
# ==== KPI-sor ====
def _eff(tab, v):
    r = tab[tab['valtozo'] == v]
    if len(r) == 0:
        return np.nan, np.nan
    b = float(r['coef'].iloc[0])
    return (np.exp(b) - 1) * 100, float(r['p'].iloc[0])

zaj_pct, zaj_p = _eff(_zaj_tab, _zaj_col)
tod_pct, tod_p = _eff(_teljes_tab, _tod_cols[0]) if _tod_cols else (np.nan, 1.0)
panel_pct, panel_p = _eff(_zaj_tab, 'is_panel')
allapot_pct, allapot_p = _eff(_zaj_tab, 'allapot_kod')
r2 = float(_meta['r2_zaj'].iloc[0])
n = int(_meta['n'].iloc[0])
rho = float(sar['sar_rho']['rho'].iloc[0]) if len(sar.get('sar_rho', [])) else np.nan
ml_r2 = float(ml['cv'].iloc[1]['R2'])

def _pct(v):
    return f"{v:+.1f}%" if pd.notna(v) else "—"

kpi_cards = [
    ("Modell R² (zaj-fókuszú)", f"{r2:.3f}", f"Kanonikus hedonikus specifikáció, N={n}", "#1e3a8a"),
    ("Vasúti zajdiszkont", _pct(zaj_pct), f"{_zaj_col} vs. {_ref} (p={zaj_p:.4f})", "#ef4444"),
    ("Kötöttpálya TOD (5p)", _pct(tod_pct), f"p = {tod_p:.4f}", "#059669"),
    ("Panel diszkont", _pct(panel_pct), f"p = {panel_p:.4f}", "#dc2626"),
    ("Állapot felár / kategória", _pct(allapot_pct), f"p = {allapot_p:.4f}", "#7c3aed"),
    ("ML térbeli CV R²", f"{ml_r2:.3f}", "Random forest, blokk-CV", "#f59e0b"),
]
display(HTML(kpi_grid_html(kpi_cards)))
if pd.notna(rho):
    display(HTML(f"<p style='color:#64748b'>SAR ρ = {rho:.3f} → térbeli multiplikátor: <b>{1/(1-rho):.2f}</b></p>"))

In [ ]:
# ==== Hatásmátrix (a kanonikus becslésekből, HC1 CI-kkel) ====
def _ci(tab, v):
    r = tab[tab['valtozo'] == v]
    if len(r) == 0:
        return np.nan, np.nan
    b = float(r['coef'].iloc[0])
    return (np.exp(float(r['ci_lo'].iloc[0])) - 1) * 100, (np.exp(float(r['ci_hi'].iloc[0])) - 1) * 100

hatasok = pd.DataFrame([
    {'Tényező': 'Panelszerkezet', 'hatas': panel_pct, 'lo': np.nan, 'hi': np.nan, 'kat': 'Fizikai'},
    {'Tényező': f'Vasúti zaj ({_zaj_col})', 'hatas': zaj_pct, 'lo': np.nan, 'hi': np.nan, 'kat': 'Környezeti'},
    {'Tényező': 'Állapotfelár (kategóriánként)', 'hatas': allapot_pct, 'lo': np.nan, 'hi': np.nan, 'kat': 'Fizikai'},
    {'Tényező': 'Kötöttpálya TOD (≤375 m)', 'hatas': tod_pct, 'lo': np.nan, 'hi': np.nan, 'kat': 'Közlekedés'},
])
for i, row in hatasok.iterrows():
    tab = _zaj_tab if row['kat'] != 'Közlekedés' else _teljes_tab
    v = {'Panelszerkezet': 'is_panel', f'Vasúti zaj ({_zaj_col})': _zaj_col,
         'Állapotfelár (kategóriánként)': 'allapot_kod', 'Kötöttpálya TOD (≤375 m)': (_tod_cols[0] if _tod_cols else None)}[row['Tényező']]
    if v:
        hatasok.loc[i, 'lo'], hatasok.loc[i, 'hi'] = _ci(tab, v)
hatasok = hatasok.sort_values('hatas')

fig_h = go.Figure()
for kat, col in [('Fizikai', '#2563eb'), ('Környezeti', '#dc2626'), ('Közlekedés', '#059669')]:
    sub = hatasok[hatasok['kat'] == kat]
    fig_h.add_trace(go.Scatter(
        x=sub['hatas'], y=sub['Tényező'], mode='markers', marker=dict(size=13, color=col),
        error_x=dict(type='data', symmetric=False,
                     array=sub['hi'] - sub['hatas'], arrayminus=sub['hatas'] - sub['lo'],
                     color=col, thickness=2, width=6), name=kat))
fig_h.add_vline(x=0, line_dash='dash', line_color='black', opacity=0.7)
fig_h.update_layout(title=f'Hatásmátrix: implicit árhatások és 95%-os HC1 CI (%) — {AREA_NEV}',
                    xaxis_title='Várható hatás a fajlagos árra (%)', template=PLOTLY_TEMPLATE, height=420)
fig_h.show()

In [ ]:
# ==== Kettős hatásgörbe: két panel, külön tengelyeken ====
# (1) ZAJLELENGÉS a vágánytávolságon (a zaj-fókuszú modell sávdummyiból)
sav_mid = {'<150 m': 75, '150-300 m': 225, '300-500 m': 400, '500-1000 m': 750,
           '1000-2000 m': 1500, '>2000 m (referencia)': 2200}
zaj_pts, zaj_x = [], []
for z in _zone_cols:
    r = _zaj_tab[_zaj_tab['valtozo'] == z]
    if len(r):
        zaj_pts.append((np.exp(float(r['coef'].iloc[0])) - 1) * 100)
        zaj_x.append(sav_mid.get(z, 1000))
# (2) TOD-LELENGÉS az állomási hálózati távolságon (a kp-dummykból, 375 m középponttal)
tod_pts, tod_x = [], []
for c in _tod_cols:
    r = _teljes_tab[_teljes_tab['valtozo'] == c]
    if len(r):
        tod_pts.append((np.exp(float(r['coef'].iloc[0])) - 1) * 100)
        mid = {'_0_375': 187, '_375_750': 562, '_750_1125': 937}.get(c[c.index('_'):c.rindex('_')] if c.count('_') > 1 else '', 500)
        tod_x.append(mid)

fig_g = make_subplots(rows=1, cols=2, subplot_titles=('Zajlecsengés a vágánytávolságon', 'TOD-lecsengés az állomástávolságon'))
fig_g.add_trace(go.Bar(x=zaj_x, y=zaj_pts, name='Becsült sávdiszkont', marker_color='#dc2626', opacity=0.75), row=1, col=1)
fig_g.add_trace(go.Bar(x=tod_x, y=tod_pts, name='Becsült TOD-prémium', marker_color='#059669', opacity=0.75), row=1, col=2)
fig_g.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4, row=1, col=1)
fig_g.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4, row=1, col=2)
fig_g.update_layout(title=f'A vasút kettős hatásgörbéje (becsült együtthatókból) — {AREA_NEV}',
                    template=PLOTLY_TEMPLATE, height=420)
fig_g.update_xaxes(title_text='Távolság a vágánytól (m)', row=1, col=1)
fig_g.update_xaxes(title_text='Hálózati távolság az állomástól (m)', row=1, col=2)
fig_g.update_yaxes(title_text='Becsült tiszta árhatás (%)', row=1, col=1)
fig_g.show()

In [ ]:
# ==== Objektív eredmény-összefoglaló (dinamikus, a cache-ből) ====
mc_sd = float(mc['meta']['resid_sd'].iloc[0]) if len(mc.get('meta', [])) else np.nan
rg_tab = rg.get('rent_gap_savok', pd.DataFrame())
rent_gap_val = float(rg_tab['median_hozam_pct'].iloc[0]) if len(rg_tab) else np.nan
lvc_tab = lvc.get('sav_diszont', pd.DataFrame())
lvc_val = float(lvc_tab['diszkont_pct'].iloc[0]) if len(lvc_tab) else np.nan

def _f(v, fmt='{:.2f}'):
    return fmt.format(v) if pd.notna(v) else '—'

html = f'''<div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:12px;padding:18px;margin:16px 0;">
<h3>Objektív eredmény-összefoglaló — {AREA_NEV}</h3>
<table style="border-collapse:collapse;width:100%;">
<tr style="background:#eef2ff;"><th style="text-align:left;padding:8px;">Mérőszám</th><th style="text-align:right;padding:8px;">Érték</th><th style="text-align:left;padding:8px;">Forrás</th></tr>
<tr><td style="padding:6px;">Kontrollált zajdiszkont ({_zaj_col} vs. {_ref})</td><td style="text-align:right;padding:6px;">{_pct(zaj_pct)}</td><td style="padding:6px;">kanonikus hedonikus modell (zaj-fókuszú, HC1)</td></tr>
<tr><td style="padding:6px;">Kötöttpálya TOD-prémium (≤375 m)</td><td style="text-align:right;padding:6px;">{_pct(tod_pct)}</td><td style="padding:6px;">kanonikus hedonikus modell (teljes)</td></tr>
<tr><td style="padding:6px;">SAR térbeli multiplikátor</td><td style="text-align:right;padding:6px;">{_f(1/(1-rho) if pd.notna(rho) and rho < 1 else np.nan)}</td><td style="padding:6px;">GM_Lag (ρ = {_f(rho, '{:.3f}')})</td></tr>
<tr><td style="padding:6px;">ML térbeli CV R²</td><td style="text-align:right;padding:6px;">{_f(ml_r2, '{:.3f}')}</td><td style="padding:6px;">random forest, 8 blokkos térbeli CV</td></tr>
<tr><td style="padding:6px;">Reziduum-szórás (árkockázat)</td><td style="text-align:right;padding:6px;">{_f(mc_sd, '{:.3f}')} ln</td><td style="padding:6px;">Monte Carlo-kalibráció</td></tr>
<tr><td style="padding:6px;">Sávonkénti bruttó hozam (első sáv)</td><td style="text-align:right;padding:6px;">{_f(rent_gap_val)}%</td><td style="padding:6px;">bérleti–eladási párosítás</td></tr>
<tr><td style="padding:6px;">Zajcsökkentés felértékelődési potenciálja ({lvc_tab['sav'].iloc[0] if len(lvc_tab) else '—'})</td><td style="text-align:right;padding:6px;">{_pct(lvc_val)}</td><td style="padding:6px;">a 03. fejezet kanonikus együtthatóiból (nincs újrabecslés)</td></tr>
</table></div>'''
display(HTML(html))

In [ ]:
# ==== Diagram- és térképgaléria: minden típus × változó-választó (JS) ====
gal_vars = {
    'nm_ar_huf': 'Négyzetméterár (HUF/m²)',
    'log_nm_ar': 'Log fajlagos ár',
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám',
    'allapot_kod': 'Állapot index (1-6)',
    'epulet_kora_ev': 'Épületkor (év)',
    'tavolsag_vasut_m': 'Vágánytávolság (m)',
    'tavolsag_vasut_halozati_m': 'Állomás hálózati táv (m)',
    'tavolsag_kotottpalya_halozati_m': 'Kötöttpálya táv (m)',
    'tavolsag_metro_halozati_m': 'Metró táv (m)',
}
_num = [c for c in gal_vars if c in p.columns and p[c].notna().sum() > 10]

figs = {}
for v in _num:
    gv = gal_vars[v]
    _d = p.dropna(subset=[v])
    figs[v] = {
        'scatter': px.scatter(_d, x='korrigalt_alapterulet_nm', y='nm_ar_huf', color=v,
                              title=f'Alapterület–ár, szín: {gv}', template=PLOTLY_TEMPLATE,
                              marginal_x='histogram'),
        'box': px.box(_d, x='vasut_zona', y=v, color='vasut_zona',
                      title=f'{gv} eloszlása immissziós sávonként', template=PLOTLY_TEMPLATE),
        'violin': px.violin(_d, x='vasut_zona', y=v, box=True,
                            title=f'{gv} — hegedűdiagram sávonként', template=PLOTLY_TEMPLATE),
        'hist': px.histogram(_d, x=v, nbins=40, title=f'{gv} hisztogram', template=PLOTLY_TEMPLATE),
        'map_scatter': px.scatter_map(_d, lat='geokodolt_lat', lon='geokodolt_lon', color=v,
                                      size='korrigalt_alapterulet_nm', size_max=12,
                                      map_style='carto-positron', zoom=11,
                                      title=f'{gv} térképen (scatter)', height=430),
        'map_density': px.density_map(_d, lat='geokodolt_lat', lon='geokodolt_lon', z=v,
                                      radius=18, map_style='carto-positron', zoom=11,
                                      title=f'{gv} sűrűségtérkép', height=430),
    }

_gal_html = '<div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:12px;padding:16px;margin:16px 0;">'
_gal_html += '<h3 style="margin:0 0 10px 0;">Diagram- és térképgaléria</h3>'
_gal_html += ('<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:12px;">'
              '<label>Változó: <select id="gal_var" style="padding:6px;">')
for c in _num:
    _gal_html += f'<option value="{c}">{gal_vars[c]}</option>'
_gal_html += '</select></label><label>Típus: <select id="gal_typ" style="padding:6px;">'
for t, tn in [('scatter', 'Szórás'), ('box', 'Doboz'), ('violin', 'Hegedű'),
              ('hist', 'Hisztogram'), ('map_scatter', 'Térkép pont'), ('map_density', 'Térkép sűrűség')]:
    _gal_html += f'<option value="{t}">{tn}</option>'
_gal_html += '</select></label></div>'
_gal_html += '<div id="gal_grid" style="display:grid;gap:12px;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));">'
for v in _num:
    for t, fig in figs[v].items():
        _gal_html += (f'<div class="gal_cell" data-var="{v}" data-typ="{t}" '
                      f'style="background:#fff;border-radius:8px;padding:6px;">{fig.to_html(full_html=False, include_plotlyjs=False)}</div>')
_gal_html += '</div>'
_gal_html += '<script>'
_gal_html += '(function(){'
_gal_html += 'var selV=document.getElementById("gal_var"), selT=document.getElementById("gal_typ");'
_gal_html += 'function upd(){'
_gal_html += 'var cells=document.querySelectorAll(".gal_cell");'
_gal_html += 'for(var i=0;i<cells.length;i++){'
_gal_html += 'var c=cells[i];'
_gal_html += 'c.style.display=(c.dataset.var===selV.value && c.dataset.typ===selT.value)?"":"none";'
_gal_html += '}}'
_gal_html += 'selV.addEventListener("change",upd); selT.addEventListener("change",upd); upd();'
_gal_html += '})();'
_gal_html += '</script></div>'
display(HTML(_gal_html))